# 06 Calibration And Fusion Review

This notebook verifies the calibration-before-fusion logic and checks whether weighted fusion improves over the uniform baseline.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/home/ubuntu/nishn_workspce/test_pdfs_generic/.covid_audio_btp_private/covid_audio_btp")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from covid_audio_btp.notebook_utils import (
    PROJECT_ROOT,
    artifact,
    check_artifacts,
    count_table,
    read_csv,
    read_optional_csv,
    require_artifacts,
    save_table,
    value_counts_frame,
    assert_no_participant_leakage,
    assert_binary_labels_present,
    stop_if_validation_errors,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
print(PROJECT_ROOT)


In [ ]:
require_artifacts([
    "data/outputs/metrics/calibration_metrics.csv",
    "data/outputs/metrics/calibrated_branch_predictions.csv",
    "data/outputs/metrics/fusion_predictions.csv",
    "data/outputs/metrics/fusion_metrics.csv",
])
calibration_metrics = read_csv("data/outputs/metrics/calibration_metrics.csv")
calibrated = read_csv("data/outputs/metrics/calibrated_branch_predictions.csv", n=5)
fusion_pred = read_csv("data/outputs/metrics/fusion_predictions.csv", n=5)
fusion_metrics = read_csv("data/outputs/metrics/fusion_metrics.csv")


## Calibration Metrics


In [ ]:
cols = [c for c in ["model_name", "modality", "calibration_method", "auroc", "auprc", "brier", "ece", "nll", "n_samples"] if c in calibration_metrics.columns]
save_table(calibration_metrics[cols], "reports/tables/nb06_calibration_metrics.csv")
display(calibration_metrics[cols].sort_values(["modality", "ece"]))

if "raw_probability" in calibrated.columns:
    delta = calibrated.assign(abs_delta=(calibrated["probability"] - calibrated["raw_probability"]).abs())
    display(delta.groupby(["model_name", "modality"])[["raw_probability", "probability", "abs_delta"]].mean().reset_index())


## Reliability Diagram


In [ ]:
def label_to_int(s):
    return s.map({"negative": 0, "positive": 1}).astype(int)

def reliability_frame(df, prob_col="probability", bins=10):
    temp = df[df["label_binary"].isin(["positive", "negative"])].copy()
    temp["bin"] = pd.cut(temp[prob_col], bins=np.linspace(0, 1, bins + 1), include_lowest=True)
    out = temp.groupby("bin", observed=False).agg(
        mean_probability=(prob_col, "mean"),
        observed_rate=("label_binary", lambda x: label_to_int(x).mean()),
        n=("label_binary", "size"),
    ).reset_index()
    return out.dropna(subset=["mean_probability", "observed_rate"])

rel = reliability_frame(calibrated)
save_table(rel, "reports/tables/nb06_reliability_bins.csv")
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], color="black", linewidth=1, linestyle="--")
plt.scatter(rel["mean_probability"], rel["observed_rate"], s=np.maximum(rel["n"], 10))
plt.xlabel("mean predicted probability")
plt.ylabel("observed positive rate")
plt.title("Branch reliability after calibration")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports/figures/nb06_reliability_diagram.png", dpi=160)
plt.show()


## Fusion Metrics And Modality Drag


In [ ]:
metric_cols = [c for c in ["fusion_method", "auroc", "auprc", "balanced_accuracy", "f1", "brier", "ece", "n_samples"] if c in fusion_metrics.columns]
save_table(fusion_metrics[metric_cols], "reports/tables/nb06_fusion_metrics.csv")
display(fusion_metrics[metric_cols].sort_values("auprc", ascending=False))

branch_best = calibration_metrics.sort_values("auprc", ascending=False).groupby("modality").head(1)
best_branch_auprc = branch_best["auprc"].max() if "auprc" in branch_best else np.nan
best_fusion_auprc = fusion_metrics["auprc"].max() if "auprc" in fusion_metrics else np.nan
print(f"best branch AUPRC={best_branch_auprc:.4f}; best fusion AUPRC={best_fusion_auprc:.4f}")
if np.isfinite(best_branch_auprc) and np.isfinite(best_fusion_auprc) and best_fusion_auprc < best_branch_auprc:
    print("Fusion underperforms the best branch. Report this as modality drag; do not claim fusion improved results.")


## Fusion Probability Checks


In [ ]:
display(fusion_pred.groupby(["fusion_method", "available_modalities"])["probability"].describe().reset_index())
if not fusion_pred["probability"].between(0, 1).all():
    raise AssertionError("Fusion probabilities outside [0, 1].")
